# Project 2: Traffic Vehicle Classification

## Reliable Traffic-Vehicle Classification with CNNs and Transfer Learning

This project moves from tabular classical machine learning to computer vision. You will build, evaluate, and critically examine an image classifier for cropped traffic-camera vehicles.

> **Scope:** All material through the end of Week 11. 

## Central Question

> Can we build a reliable system that classifies vehicles from traffic-camera images, explains its failures, and recommends human review for uncertain predictions?

The goal is not only accuracy. The final system should be reproducible, robust to data-quality issues, and transparent about uncertainty.

## Dataset

Each image is a crop of one vehicle from a traffic-camera frame. The expected layout is:

| Split | Classes | Intended use |
|---|---|---|
| `dataset/train` | `ambulance`, `autobus`, `kamyun`, `kamyunet`, `minibus`, `savari`, `taxi`, `vanet` | Training and validation source |
| `dataset/test` | The same 8 classes | Final frozen evaluation |
| `dataset/unclean` | The 8 known classes plus `neysan` | Data-quality and unseen-class analysis |

The supplied listing suggests roughly 400 train, 400 test, and 450 unclean images. Verify all counts programmatically; do not hard-code them. The original train split appears class-balanced.

## Data Quality and Evaluation Protocol

Do **not** merge `unclean` directly into training. The supplied listing contains repeated filenames across splits and at least one apparent label conflict. Filename equality is not enough: use an image-content hash to identify real duplicates.

Follow this order:

1. Find unreadable files, unsupported formats, unusually small images, and unusual image dimensions.
2. Report class counts, image-size distributions, and representative examples.
3. Detect duplicate images with hashes across `train`, `test`, and `unclean`; document conflicts.
4. Create cleaned splits and document every exclusion decision.
5. Create a stratified validation split from cleaned training data using a fixed seed; 80/20 is a recommended starting point.
6. Freeze the cleaned test set. Use validation for all model, threshold, scheduler, and hyperparameter choices; use test once for final evaluation.

## Learning Outcomes

By the end of this project, you should be able to:

- Audit an image dataset and load it with `ImageFolder` and `DataLoader`.
- Design transforms for resize, normalization, and augmentation.
- Build a CNN baseline and a complete PyTorch training loop.
- Implement and evaluate a custom `BalancedBatchSampler`.
- Compare `CrossEntropyLoss` and `BCEWithLogitsLoss` for a single-label task.
- Evaluate dropout, weight decay, pooling, augmentation, and learning-rate scheduling.
- Use pretrained ResNet18 for feature extraction and fine-tuning.
- Compare models with per-class precision, recall, F1, and confusion matrices.
- Analyze errors, class relationships, and low-confidence predictions.
- Deliver a reproducible prediction script with JSON output.

## Required Technical Work

### 1. Data audit and baseline
- Report counts for original and cleaned splits, and display at least 12 labelled images.
- Record the random seed, validation indices, and every transform.
- Implement a CNN with at least two convolution + activation + pooling blocks.
- Record training loss, validation loss, and validation metrics at every epoch. Save the best checkpoint by a validation metric.

### 2. Controlled ablations
Change one factor at a time, unless the goal is explicitly to test a combination. Use the same baseline, split, and training budget for each one-factor comparison.

| Experiment | Question |
|---|---|
| No augmentation vs. augmentation | Does realistic variation improve generalization? |
| Dropout: `p=0`, `0.3`, or `0.5` | Does dropout reduce overfitting? |
| Max pooling vs. average pooling | Which spatial summary is more useful? |
| `weight_decay=0` vs. `1e-4` | Does weight regularization improve validation behaviour? |
| Fixed learning rate vs. scheduler | Does scheduling improve convergence or validation performance? |
| Standard batches vs. balanced batches | How does sampling affect recall under imbalance? |
| Cross-entropy vs. BCE | How do the loss assumptions affect a single-label problem? |
| Feature extraction vs. fine-tuning | Does adapting a pretrained backbone help? |

## Loading the Dataset Correctly

Do not use `ImageFolder("dataset")`: it would interpret `train`, `test`, and `unclean` as class names. Load each split separately:

```python
from torchvision import datasets

train_dataset = datasets.ImageFolder("dataset/train", transform=train_transform)
test_dataset = datasets.ImageFolder("dataset/test", transform=eval_transform)
assert train_dataset.class_to_idx == test_dataset.class_to_idx
```

Load `unclean` separately. Because it contains `neysan`, it is not an ordinary 8-class test split.

## Balanced Batches on Simulated Imbalance

The original train split appears balanced, so a balanced sampler would have little visible effect. Create a **reproducible simulated imbalance** from cleaned training data: with a fixed seed, retain fewer examples from selected classes and report the retained indices. Do not use `unclean` to create imbalance.

Implement a custom `BalancedBatchSampler` that places an equal number of examples from all 8 classes in every batch. `batch_size` must be divisible by 8. Sampling with replacement is allowed for smaller classes. When passing `batch_sampler` to `DataLoader`, do not also pass `batch_size`, `shuffle`, or a separate `sampler`.

Compare it with standard `shuffle=True` batches on the same simulated-imbalanced training set. Keep validation and test unchanged, and compare accuracy, macro-F1, and recall for every class.

## Cross-Entropy vs. Binary Cross-Entropy

For the primary controlled comparison, keep architecture, seed, split, augmentation, optimizer, learning rate, and training budget fixed. If you later compare the best-tuned version of each loss, give both losses equal tuning budgets.

- `CrossEntropyLoss` is the standard choice: the target is an integer class index and the objective models one mutually exclusive class decision.
- `BCEWithLogitsLoss` is an educational one-vs-all experiment: the target must be one-hot encoded and `float`, representing eight independent yes/no decisions.

For BCE, use `argmax(logits)` to get one class for multiclass metrics. Do not apply sigmoid separately before `BCEWithLogitsLoss`. Explain why BCE scores are not necessarily mutually exclusive and do not necessarily sum to one. Compare loss curves, per-class metrics, confidence behaviour, and error patterns.

## Regularization and Learning-Rate Scheduling

Study dropout, weight decay, augmentation, and a scheduler through controlled experiments. Use `AdamW` for the weight-decay experiment when possible. Compare a fixed learning rate with `StepLR` or `ReduceLROnPlateau`.

For each run, report training/validation loss, the train-validation gap, best epoch, macro-F1, and learning rate at every epoch. With `ReduceLROnPlateau`, call `scheduler.step(validation_loss)` after validation. Test performance must not drive any training decision.

## Transfer Learning with ResNet18

Compare these required strategies:

1. **Feature extraction:** freeze an ImageNet-pretrained ResNet18 backbone and train a new 8-class head.
2. **Fine-tuning:** train the head first, then unfreeze `layer4` and the head with a smaller learning rate for pretrained parameters.

For these pretrained comparisons, keep the standard ResNet18 architecture and use ImageNet-compatible resize and normalization. Do not replace `conv1` or remove `maxpool`, because the pretrained first-layer weights would no longer match.

Stretch: train `ResNet18(weights=None)` for small images. In that separate experiment, changing `conv1` and removing `maxpool` is acceptable. Report trainable parameters and learning-rate groups for all transfer-learning runs.

## Evaluation, Precision, Recall, and Error Analysis

Use validation data for experiment comparison. After choosing the final configuration, evaluate it once on the frozen test set. Report overall accuracy, macro precision, macro recall, macro-F1, per-class precision/recall/F1, count-based and row-normalized confusion matrices, training curves, and at least 12 misclassified images with true label, prediction, and confidence.

Create this validation comparison table for the main experiments, then analyze the lowest-precision and lowest-recall class in each meaningful case:

| Experiment | Macro Precision | Macro Recall | Macro F1 | Lowest-Recall Class | Lowest-Precision Class |
|---|---:|---:|---:|---|---|
| CNN baseline |  |  |  |  |  |
| Balanced batches |  |  |  |  |  |
| CrossEntropy / BCE comparison |  |  |  |  |  |
| Best regularized + scheduled model |  |  |  |  |  |
| ResNet feature extraction |  |  |  |  |  |
| ResNet fine-tuning |  |  |  |  |  |

## Class Relationships, `unclean`, and Confidence

Use the row-normalized confusion matrix to rank mutually confused class pairs. A useful starting score is:

```text
pair_confusion(i, j) = C_normalized[i, j] + C_normalized[j, i]
```

Inspect the actual errors, class support, and the practical meaning of labels before proposing a merge. If a pair has substantial mutual confusion and a domain justification, define a merged label mapping, retrain, and compare the original and merged taxonomies. If not, defend the decision not to merge. Retraining a justified merged taxonomy is stretch work.

Do not train on `unclean` initially. First exclude duplicates of train/test images, then inspect label conflicts. Analyze `neysan` as an unseen vehicle class or a human-review case, not as an unexplained ninth training class. Propose `needs_review=True` below a confidence threshold such as `0.70`, but choose and justify the threshold with validation analysis, never with test data.

## Prediction Output and Reproducibility

Implement `predict.py` or an equivalent callable function that accepts an image path and returns JSON:

```json
{
  "predicted_class": "taxi",
  "confidence": 0.87,
  "probabilities": {"ambulance": 0.01, "autobus": 0.02, "kamyun": 0.03, "kamyunet": 0.02, "minibus": 0.01, "savari": 0.02, "taxi": 0.87, "vanet": 0.02},
  "needs_review": false
}
```

The production output should use the selected Cross-Entropy model, whose softmax probabilities sum to approximately one. Keep BCE sigmoid scores in the comparison report rather than presenting them as a final probability distribution. Save checkpoint metadata: architecture, class mapping, transforms, selected threshold, seed, and experiment configuration.

## Deliverables and Grading

Submit a Git repository with a clear README, requirements, reproducible configuration, data-audit report, baseline CNN, controlled ablations, sampler implementation, CE/BCE report, scheduler report, pretrained ResNet comparison, confusion matrices, per-class precision/recall/F1 analysis, `unclean` analysis, and JSON prediction script. Keep datasets and large checkpoints out of Git.

| Component | Points |
|---|---:|
| Repository structure, Git, and README | 5 |
| Data audit and leakage prevention | 10 |
| CNN baseline and training loop | 10 |
| Balanced batches and imbalance analysis | 10 |
| Cross-Entropy vs. BCE comparison | 10 |
| Regularization and learning-rate scheduler | 10 |
| ResNet transfer learning | 20 |
| Evaluation, per-class analysis, confusion matrices, and merge decision | 10 |
| `unclean` and low-confidence analysis | 5 |
| Prediction script, JSON, and reproducibility | 10 |
| **Total** | **100** |

Stretch work: ResNet18 from scratch for small images, retraining a justified merged taxonomy, calibration analysis, mixed precision, Grad-CAM, a FastAPI endpoint, or unit tests.

## Final Checklist

- [ ] Is the class mapping identical between train and test?
- [ ] Was the audit completed before creating validation data and freezing the cleaned test set?
- [ ] Was test used only once for final evaluation?
- [ ] Were duplicates across splits checked with image hashes?
- [ ] Is augmentation applied only to training data?
- [ ] Was the balanced-batch experiment run on a reproducible simulated-imbalanced training set?
- [ ] Does BCE use one-hot `float` targets and `argmax(logits)` for multiclass metrics?
- [ ] Does the scheduler use validation data only, and are learning rates logged?
- [ ] Are accuracy, precision, recall, F1, and both confusion-matrix forms reported?
- [ ] Are per-class precision, recall, and F1 compared across the main experiments?
- [ ] Are the lowest-precision and lowest-recall classes analyzed?
- [ ] Were misclassified images inspected and explained?
- [ ] Was `unclean` analyzed separately after duplicate removal?
- [ ] Does the checkpoint contain class mapping, transforms, threshold, seed, and configuration?
- [ ] Can another person reproduce the final result from the README?